In [ ]:
import numpy as np
import torch, torch.nn as nn, torch.optim as optim
import bidsio
import sys
import copy
import pickle
sys.path.append('../')
from helpers import *
from fns_gap import *
from bandlimited_signal import *


device = 'cuda:5' if torch.cuda.is_available() else 'cpu'
seed, RES = 0, 64


In [ ]:
def load_gt(dataset, id_val, bandlimit = 0.6):
    RES = 64
    if dataset=='dragon':
        signal =  resize(Voxel_Fitting(dimension=3, length=RES, bandlimit=bandlimit, seed=id_val, super_resolution=False, sparse=True).signal, (RES, RES, RES))
    elif dataset == 'sphere':
        signal = SparseSphereSignal(dimension=3, length=RES, bandlimit=bandlimit, seed=id_val, generate=False).signal
    elif dataset == 'bandlimited':
        signal = BandlimitedSignal(dimension=3, length=RES, bandlimit=bandlimit, seed=id_val, generate=False).signal  
    return signal

dataset='dragon'
id_val = 1234
signal = load_gt(dataset, id_val)
print(signal.shape)
plt.imshow(signal[:,:,32], cmap="gray")
plt.colorbar()
plt.show()

In [ ]:
learning_rate, iters = 8e-3, 4000
slice_idx = 32

# varying volume resolution
# model_params = {
#     "lineres": 2,
#     "planeres": 2,
#     "volumeres": -1, 
#     "line_feature_dim": 4, 
#     "plane_feature_dim": 4,
#     "volume_feature_dim": 1,
#     "hidden_dim": 64,
#     "operation": 'concatenate',
#     "decoder": 'linear',
# }
# param_vals = [43, 54]

# varying feature dimension
model_params = {
    "lineres": 64,
    "planeres": 64,
    "volumeres": 2, 
    "line_feature_dim": -1, 
    "plane_feature_dim": -1,
    "volume_feature_dim": 1,
    "hidden_dim": 64,
    "operation": 'concatenate',
    "decoder": 'linear',
}
param_vals = [4, 9]
if model_params['line_feature_dim'] == -1 and model_params['plane_feature_dim'] == -1:
    param_to_vary = 'feature_dim'
else:    
    param_to_vary = 'volumeres'
outputs = {}
to_save_outputs = {}
for param_val in param_vals:
    if param_to_vary == 'feature_dim':
        model_params['line_feature_dim'] = param_val
        model_params['plane_feature_dim'] = param_val
        print(f'line_feature_dim: {model_params["line_feature_dim"]}, plane_feature_dim: {model_params["plane_feature_dim"]}')
    else:
        model_params[param_to_vary] = param_val
        print(f'{param_to_vary}: {model_params[param_to_vary]}')
    print(model_params)
    print("Manual compute: ", compute_model_size("ga-planes_eta", model_params, n_dims=3))
    output = fit_gaplanes(args=model_params, img=signal, iters=iters, learning_rate=learning_rate, log_interval = 1000, seed=seed, device=device, count_params=True)
    if param_to_vary == 'feature_dim':
        outputs[f'{model_params['line_feature_dim']}'] = output     
        to_save_outputs[f'{model_params['line_feature_dim']}'] = output['best_pred'].reshape((RES, RES, RES))[:,:,slice_idx]
    else:
        outputs[f'{model_params[param_to_vary]}'] = output
        to_save_outputs[f'{model_params[param_to_vary]}'] = output['best_pred'].reshape((RES, RES, RES))[:,:,slice_idx]
    error = np.linalg.norm(signal.flatten() - output['best_pred'].flatten())                       
    print(f"Error: {error:.3e}, Loss: {output['best_loss']:.3e}")

with open(f"3d_dragon/gap.pkl", "wb") as f:
    pickle.dump(to_save_outputs, f)

In [ ]:
from helpers import *

slice_idx = 32
def slice_outputs(outputs, slice_idx):
    new_outputs = copy.deepcopy(outputs)
    small_param, large_param = list(outputs.keys())
    small_pred, large_pred   = new_outputs[small_param]['best_pred'].reshape((RES, RES, RES)), new_outputs[large_param]['best_pred'].reshape((RES, RES, RES))
    new_outputs[small_param], new_outputs[large_param] = new_outputs[small_param], new_outputs[large_param]
    new_outputs[small_param]['best_pred'] = small_pred[:,:,slice_idx]
    new_outputs[large_param]['best_pred'] = large_pred[:,:,slice_idx]
    return new_outputs

plot_error_heatmaps(signal[:,:,slice_idx], slice_outputs(outputs, slice_idx), model_name="GA-Planes")